#  Hospital Patient Care Analytics Pipeline
### End-to-End ETL (Extract, Transform, Load) & Clinical Analytics System

**Author:** Data Engineering Fundamentals Student  
**Stack:** Python, Pandas, SQLite, SQL, CSV  

---

##  Scenario Overview
A multi-specialty hospital collects patient information from multiple isolated clinical systems:
1. **Patient Registration** (`patients.csv`): Demographics and chronic medical history.
2. **Appointment Scheduling** (`appointments.csv`): Department bookings and scheduled appointment slots.
3. **Doctor Consultations** (`consultations.csv`): Actual consultation times, diagnoses, and doctor notes.
4. **Laboratory Reports** (`lab_reports.csv`): Diagnostic tests, numerical results, and clinical reference ranges.
5. **Wearable Health Telemetry** (`wearable_data.csv`): Continuous streams of heart rate, blood pressure, SpO2, and activity.

### Project Goals
- Ingest and standardize multi-source clinical data using **Pandas**.
- Clean duplicates, impute missing values using **median and mode techniques** (extending `ETLPipeline.ipynb`).
- Calculate operational metrics: **Patient Waiting Time** (Scheduled Time vs. Consultation Time).
- Implement a point-based **Clinical Risk Scoring Model** based on vital sign thresholds (Heart Rate, Blood Pressure, SpO2, Lab Abnormalities, Comorbidities).
- Load cleaned dimensional tables and master analytical views into **SQLite**.
- Execute **SQL Analytical Queries** to uncover operational bottlenecks and identify high-risk patients requiring immediate intervention.

## 0. Environment Setup & Library Imports

In [33]:
import os
import sqlite3
import pandas as pd
import numpy as np

# Display formatting for clean presentation tables
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

print("Libraries imported successfully. Ready for ETL Pipeline execution.")

Libraries imported successfully. Ready for ETL Pipeline execution.


## 1. EXTRACT: Ingesting Raw Data
We extract the 5 raw CSV datasets from `data/raw/` into Pandas DataFrames and inspect their structure.

In [34]:
# File paths
raw_dir = os.path.join("data", "raw")

df_patients = pd.read_csv(os.path.join(raw_dir, "patients.csv"))
df_appointments = pd.read_csv(os.path.join(raw_dir, "appointments.csv"))
df_consultations = pd.read_csv(os.path.join(raw_dir, "consultations.csv"))
df_labs = pd.read_csv(os.path.join(raw_dir, "lab_reports.csv"))
df_wearables = pd.read_csv(os.path.join(raw_dir, "wearable_data.csv"))

print(f"Patients:      {df_patients.shape[0]} rows, {df_patients.shape[1]} columns")
print(f"Appointments:  {df_appointments.shape[0]} rows, {df_appointments.shape[1]} columns")
print(f"Consultations: {df_consultations.shape[0]} rows, {df_consultations.shape[1]} columns")
print(f"Lab Reports:   {df_labs.shape[0]} rows, {df_labs.shape[1]} columns")
print(f"Wearables:     {df_wearables.shape[0]} rows, {df_wearables.shape[1]} columns")

Patients:      16 rows, 8 columns
Appointments:  16 rows, 7 columns
Consultations: 16 rows, 7 columns
Lab Reports:   25 rows, 8 columns
Wearables:     30 rows, 8 columns


In [35]:
# Inspect sample records
print("--- Patients Sample ---")
display(df_patients.head(3))

print("--- Wearable Telemetry Sample ---")
display(df_wearables.head(3))

--- Patients Sample ---


,patient_id,first_name,last_name,gender,date_of_birth,phone,blood_group,chronic_condition
0,P001,John,Doe,Male,1958-03-15,555-0101,O+,Hypertension
1,P002,Jane,Smith,female,1975-08-22,555-0102,A-,Diabetes
2,P003,Robert,Johnson,M,1948-11-05,NaN,B+,Coronary Artery Disease


--- Wearable Telemetry Sample ---


,device_id,patient_id,timestamp,heart_rate,systolic_bp,diastolic_bp,spo2,daily_steps
0,DEV_101,P001,2025-05-10 08:00:00,78,142,92,97.0,1420
1,DEV_101,P001,2025-05-10 08:30:00,82,145,95,96.0,2100
2,DEV_101,P001,2025-05-10 09:00:00,80,140,90,97.0,2850


## 2. DATA QUALITY AUDIT (Auditing Missing Values & Duplicates)
Just as demonstrated in `ETLPipeline.ipynb` using `.isnull().sum()` and checking duplicates, we assess the raw data quality.

In [36]:
print("=== Missing Values Audit ===")
print("\nPatients Missing Values:")
print(df_patients.isnull().sum()[df_patients.isnull().sum() > 0])

print("\nWearable Telemetry Missing Values:")
print(df_wearables.isnull().sum()[df_wearables.isnull().sum() > 0])

print("\n=== Duplicate Records Audit ===")
print(f"Duplicate patient rows:      {df_patients.duplicated().sum()}")
print(f"Duplicate appointment rows:  {df_appointments.duplicated().sum()}")
print(f"Duplicate consultation rows: {df_consultations.duplicated().sum()}")
print(f"Duplicate lab rows:          {df_labs.duplicated().sum()}")
print(f"Duplicate wearable rows:     {df_wearables.duplicated().sum()}")

=== Missing Values Audit ===

Patients Missing Values:
phone                3
chronic_condition    5
dtype: int64

Wearable Telemetry Missing Values:
spo2    1
dtype: int64

=== Duplicate Records Audit ===
Duplicate patient rows:      1
Duplicate appointment rows:  1
Duplicate consultation rows: 1
Duplicate lab rows:          1
Duplicate wearable rows:     1


## 3. TRANSFORM: Data Cleaning & Standardization

### 3.1 Deduplication & Standardizing Categorical Fields
- Remove duplicate records across all datasets.
- Standardize casing and strip whitespace.
- Standardize gender (`'M'`, `'male'` $\to$ `'Male'`).
- Handle missing values: Fill phone with `'Not Provided'` and impute missing numerical telemetry with **median** values.

In [49]:
# 1. Deduplicate
df_patients.drop_duplicates(inplace=True)
df_appointments.drop_duplicates(inplace=True)
df_consultations.drop_duplicates(inplace=True)
df_labs.drop_duplicates(inplace=True)
df_wearables.drop_duplicates(inplace=True)

# 2. Standardize Patients
df_patients["first_name"] = df_patients["first_name"].str.strip().str.title()
df_patients["last_name"] = df_patients["last_name"].str.strip().str.title()
df_patients["full_name"] = df_patients["first_name"] + " " + df_patients["last_name"]

gender_map = {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
df_patients["gender"] = df_patients["gender"].astype(str).str.strip().str.lower().map(gender_map).fillna("Other")
df_patients["phone"] = df_patients["phone"].fillna("Not Provided")
df_patients["chronic_condition"] = df_patients["chronic_condition"].fillna("None")

# Calculate Patient Age from DOB
df_patients["date_of_birth"] = pd.to_datetime(df_patients["date_of_birth"])
ref_date = pd.Timestamp("2025-05-10")
df_patients["age"] = ((ref_date - df_patients["date_of_birth"]).dt.days // 365).astype(int)
df_patients["date_of_birth"] = df_patients["date_of_birth"].dt.strftime("%Y-%m-%d")

# 3. Standardize Appointments
df_appointments["department"] = df_appointments["department"].str.strip().str.title()
df_appointments["status"] = df_appointments["status"].str.strip().str.title()
df_appointments["scheduled_time"] = pd.to_datetime(df_appointments["scheduled_time"])

# 4. Wearables Missing Value Imputation using Median (from ETLPipeline.ipynb pattern)
median_spo2 = df_wearables["spo2"].median()
median_steps = df_wearables["daily_steps"].median()
df_wearables["spo2"] = df_wearables["spo2"].fillna(median_spo2)
df_wearables["daily_steps"] = df_wearables["daily_steps"].fillna(median_steps)

print(" Cleaning and median imputation complete. Zero nulls remaining in critical fields.")

 Cleaning and median imputation complete. Zero nulls remaining in critical fields.


### 3.2 Operational Metrics: Calculating Patient Waiting Time
$$\text{Waiting Time (minutes)} = \frac{\text{Consultation Time} - \text{Scheduled Appointment Time}}{60 \text{ seconds}}$$

In [38]:
df_consultations["consultation_time"] = pd.to_datetime(df_consultations["consultation_time"])

# Merge consultations with scheduled_time
merged_consult = pd.merge(
    df_consultations,
    df_appointments[["appointment_id", "scheduled_time"]],
    on="appointment_id",
    how="left"
)

# Calculate waiting time in minutes
df_consultations["waiting_time_minutes"] = (
    (merged_consult["consultation_time"] - merged_consult["scheduled_time"]).dt.total_seconds() / 60.0
).round(1).clip(lower=0.0)

# Format timestamps back to ISO strings
df_consultations["consultation_time"] = df_consultations["consultation_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_appointments["scheduled_time"] = df_appointments["scheduled_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

print("Sample Calculated Waiting Times:")
display(df_consultations[["consultation_id", "appointment_id", "waiting_time_minutes", "diagnosis"]].head(5))

Sample Calculated Waiting Times:


,consultation_id,appointment_id,waiting_time_minutes,diagnosis
0,C201,A101,25.0,Stage 2 Hypertension
1,C202,A102,40.0,Type 2 Diabetes
2,C203,A103,75.0,Unstable Angina
3,C204,A104,15.0,Lumbar Muscle Strain
4,C205,A105,65.0,Acute COPD Exacerbation


### 3.3 Diagnostic Metrics: Flagging Abnormal Lab Results
$$\text{is\_abnormal} = 1 \quad \text{if } (\text{test\_value} < \text{reference\_min} \text{ OR } \text{test\_value} > \text{reference\_max}), \quad \text{else } 0$$

In [39]:
df_labs["test_value"] = pd.to_numeric(df_labs["test_value"], errors="coerce")
df_labs["reference_min"] = pd.to_numeric(df_labs["reference_min"], errors="coerce")
df_labs["reference_max"] = pd.to_numeric(df_labs["reference_max"], errors="coerce")

df_labs["is_abnormal"] = (
    (df_labs["test_value"] < df_labs["reference_min"]) |
    (df_labs["test_value"] > df_labs["reference_max"])
).astype(int)

# Aggregate lab metrics by patient
patient_labs = df_labs.groupby("patient_id").agg(
    total_lab_tests=("lab_id", "count"),
    abnormal_lab_count=("is_abnormal", "sum")
).reset_index()

display(patient_labs.head(5))

,patient_id,total_lab_tests,abnormal_lab_count
0,P001,2,2
1,P002,2,1
2,P003,2,2
3,P004,1,0
4,P005,2,1


### 3.4 Telemetry Aggregation: Patient Resting Vitals
Summarize multiple wearable readings per patient to obtain resting baseline vitals.

In [40]:
patient_wearables = df_wearables.groupby("patient_id").agg(
    avg_heart_rate=("heart_rate", lambda x: round(x.mean(), 1)),
    avg_systolic_bp=("systolic_bp", lambda x: round(x.mean(), 1)),
    avg_diastolic_bp=("diastolic_bp", lambda x: round(x.mean(), 1)),
    min_spo2=("spo2", lambda x: round(x.min(), 1))
).reset_index()

display(patient_wearables.head(5))

,patient_id,avg_heart_rate,avg_systolic_bp,avg_diastolic_bp,min_spo2
0,P001,80.0,142.3,92.3,96.0
1,P002,73.5,129.0,83.0,98.0
2,P003,118.0,171.7,105.0,89.0
3,P004,69.0,119.0,77.0,99.0
4,P005,98.0,138.7,88.0,90.0


### 3.5 Clinical Risk Scoring Rules & Thresholds Engine

To move beyond a simplistic high-risk flag, we implement a **clinical early warning point-scoring model**:

| Clinical Metric | Normal (0 pts) | Mild (1 pt) | Moderate (2 pts) | Severe (3 pts) |
| :--- | :--- | :--- | :--- | :--- |
| **Heart Rate (bpm)** | 60 - 100 | 51-59 or 101-110 | 41-50 or 111-130 | $\le 40$ or $> 130$ |
| **Systolic BP (mmHg)** | 110 - 139 | 100-109 or 140-159 | 91-99 or 160-179 | $\le 90$ or $\ge 180$ |
| **Oxygen Saturation (SpO2)** | $\ge 96\%$ | 94% - 95% | 92% - 93% | $< 92\%$ |
| **Abnormal Lab Tests** | 0 | 1 | 2 | $\ge 3$ |
| **Comorbidity / Age** | Age < 65 & None | Age $\ge 65$ OR Moderate | High-Risk Disease (Heart Failure/COPD/CKD) | - |

**Stratification Tiers:**
- **Low Risk:** Total Score 0 - 2 (Routine outpatient monitoring)
- **Moderate Risk:** Total Score 3 - 5 (Priority clinic monitoring)
- **High Risk:** Total Score $\ge 6$ OR any single vital score == 3 (Immediate emergency/consultant escalation)

In [41]:
# Clinical scoring rule definitions
def score_hr(hr):
    if 60 <= hr <= 100: return 0
    elif (51 <= hr < 60) or (101 <= hr <= 110): return 1
    elif (41 <= hr <= 50) or (111 <= hr <= 130): return 2
    return 3

def score_bp(sbp):
    if 110 <= sbp <= 139: return 0
    elif (100 <= sbp < 110) or (140 <= sbp <= 159): return 1
    elif (91 <= sbp < 100) or (160 <= sbp <= 179): return 2
    return 3

def score_spo2(spo2):
    if spo2 >= 96.0: return 0
    elif 94.0 <= spo2 < 96.0: return 1
    elif 92.0 <= spo2 < 94.0: return 2
    return 3

def score_labs(abnormal_cnt):
    if abnormal_cnt == 0: return 0
    elif abnormal_cnt == 1: return 1
    elif abnormal_cnt == 2: return 2
    return 3

def score_comorbidity(cond, age):
    pts = 1 if age >= 65 else 0
    cond = str(cond).lower()
    if any(k in cond for k in ["heart failure", "copd", "chronic kidney disease", "coronary artery disease"]):
        pts += 2
    elif any(k in cond for k in ["hypertension", "diabetes", "arrhythmia", "asthma"]):
        pts += 1
    return pts

# Build Master Patient Health Summary Table
df_summary = pd.merge(
    df_patients[["patient_id", "full_name", "age", "gender", "chronic_condition"]],
    df_appointments[["appointment_id", "patient_id", "department"]],
    on="patient_id", how="left"
)

df_summary = pd.merge(
    df_summary,
    df_consultations[["appointment_id", "waiting_time_minutes", "diagnosis"]],
    on="appointment_id", how="left"
)

df_summary = pd.merge(df_summary, patient_wearables, on="patient_id", how="left")
df_summary = pd.merge(df_summary, patient_labs, on="patient_id", how="left")

df_summary["total_lab_tests"] = df_summary["total_lab_tests"].fillna(0).astype(int)
df_summary["abnormal_lab_count"] = df_summary["abnormal_lab_count"].fillna(0).astype(int)

# Apply Scoring Rules
df_summary["hr_risk_score"] = df_summary["avg_heart_rate"].apply(score_hr)
df_summary["bp_risk_score"] = df_summary["avg_systolic_bp"].apply(score_bp)
df_summary["spo2_risk_score"] = df_summary["min_spo2"].apply(score_spo2)
df_summary["lab_risk_score"] = df_summary["abnormal_lab_count"].apply(score_labs)
df_summary["comorbidity_risk_score"] = df_summary.apply(lambda r: score_comorbidity(r["chronic_condition"], r["age"]), axis=1)

df_summary["total_clinical_risk_score"] = (
    df_summary["hr_risk_score"] +
    df_summary["bp_risk_score"] +
    df_summary["spo2_risk_score"] +
    df_summary["lab_risk_score"] +
    df_summary["comorbidity_risk_score"]
)

max_vital_subscore = df_summary[["hr_risk_score", "bp_risk_score", "spo2_risk_score", "lab_risk_score"]].max(axis=1)

def classify_risk(row, max_sub):
    score = row["total_clinical_risk_score"]
    if score >= 6 or max_sub >= 3:
        return pd.Series(["High", 1, "URGENT: Escalate to Senior Consultant / High-Dependency Unit"])
    elif score >= 3:
        return pd.Series(["Moderate", 0, "PRIORITY: Continuous vitals monitoring & clinical review"])
    else:
        return pd.Series(["Low", 0, "ROUTINE: Standard outpatient care & scheduled follow-up"])

df_summary[["clinical_risk_tier", "high_risk_flag", "care_recommendation"]] = df_summary.apply(
    lambda r: classify_risk(r, max_vital_subscore.loc[r.name]), axis=1
)

df_summary.rename(columns={"full_name": "patient_name"}, inplace=True)
print(" Patient Health Summary and Clinical Risk Scoring Matrix Built Successfully.")
display(df_summary[["patient_id", "patient_name", "department", "total_clinical_risk_score", "clinical_risk_tier", "high_risk_flag"]].head(8))

 Patient Health Summary and Clinical Risk Scoring Matrix Built Successfully.


,patient_id,patient_name,department,total_clinical_risk_score,clinical_risk_tier,high_risk_flag
0,P001,John Doe,Cardiology,5,Moderate,0
1,P002,Jane Smith,General Medicine,2,Low,0
2,P003,Robert Johnson,Cardiology,12,High,1
3,P004,Emily Davis,Orthopedics,0,Low,0
4,P005,Michael Brown,Pulmonology,6,High,1
5,P006,Sarah Wilson,Pediatrics,0,Low,0
6,P007,David Martinez,Cardiology,10,High,1
7,P008,Lisa Taylor,Pulmonology,2,Low,0


## 4. LOAD: Persisting to SQLite Database & Processed CSVs
We store all 6 tables in our local **SQLite** database (`database/hospital_analytics.db`) using Python's native `sqlite3` and `DataFrame.to_sql()`.

In [42]:
db_dir = "database"
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "hospital_analytics.db")

conn = sqlite3.connect(db_path)

# Load clean tables to SQLite
tables = {
    "patients": df_patients,
    "appointments": df_appointments,
    "consultations": df_consultations,
    "lab_reports": df_labs,
    "wearable_data": df_wearables,
    "patient_health_summary": df_summary
}

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)
    count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"Loaded table '{name}': {count} rows in SQLite.")

# Also save sanitized CSV files to data/processed/
proc_dir = os.path.join("data", "processed")
os.makedirs(proc_dir, exist_ok=True)
for name, df in tables.items():
    fname = f"cleaned_{name}.csv" if name != "patient_health_summary" else "patient_health_summary.csv"
    df.to_csv(os.path.join(proc_dir, fname), index=False)

print("\n All tables loaded into SQLite and exported to data/processed/")

Loaded table 'patients': 15 rows in SQLite.
Loaded table 'appointments': 15 rows in SQLite.
Loaded table 'consultations': 15 rows in SQLite.
Loaded table 'lab_reports': 24 rows in SQLite.
Loaded table 'wearable_data': 29 rows in SQLite.
Loaded table 'patient_health_summary': 15 rows in SQLite.

 All tables loaded into SQLite and exported to data/processed/


## 5. ANALYTICS: SQL Queries & Hospital KPIs
Now we run SQL queries against `hospital_analytics.db` to generate operational and clinical insights.

### KPI 1: Average Patient Waiting Time by Department

In [43]:
q1 = """
SELECT 
    a.department,
    COUNT(c.consultation_id) AS total_patients,
    ROUND(AVG(c.waiting_time_minutes), 1) AS avg_wait_mins,
    ROUND(MIN(c.waiting_time_minutes), 1) AS min_wait_mins,
    ROUND(MAX(c.waiting_time_minutes), 1) AS max_wait_mins
FROM appointments a
JOIN consultations c ON a.appointment_id = c.appointment_id
GROUP BY a.department
ORDER BY avg_wait_mins DESC;
"""
df_kpi1 = pd.read_sql_query(q1, conn)
display(df_kpi1)

,department,total_patients,avg_wait_mins,min_wait_mins,max_wait_mins
0,Nephrology,1,65.0,65.0,65.0
1,Cardiology,5,47.0,25.0,75.0
2,Pulmonology,2,42.5,20.0,65.0
3,General Medicine,3,24.0,12.0,40.0
4,Neurology,1,20.0,20.0,20.0
5,Pediatrics,1,18.0,18.0,18.0
6,Orthopedics,2,16.5,15.0,18.0


### KPI 2: Appointment Volume Breakdown by Department

In [44]:
q2 = """
SELECT 
    department,
    COUNT(*) AS total_appointments,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM appointments), 1) AS pct_share
FROM appointments
GROUP BY department
ORDER BY total_appointments DESC;
"""
df_kpi2 = pd.read_sql_query(q2, conn)
display(df_kpi2)

,department,total_appointments,pct_share
0,Cardiology,5,33.3
1,General Medicine,3,20.0
2,Pulmonology,2,13.3
3,Orthopedics,2,13.3
4,Pediatrics,1,6.7
5,Neurology,1,6.7
6,Nephrology,1,6.7


### KPI 3: Frequency of Abnormal Lab Findings

In [45]:
q3 = """
SELECT 
    test_name,
    COUNT(*) AS total_tests,
    SUM(is_abnormal) AS abnormal_count,
    ROUND(100.0 * SUM(is_abnormal) / COUNT(*), 1) AS abnormal_rate_pct,
    ROUND(AVG(test_value), 2) AS avg_test_value,
    test_unit
FROM lab_reports
GROUP BY test_name, test_unit
ORDER BY abnormal_count DESC;
"""
df_kpi3 = pd.read_sql_query(q3, conn)
display(df_kpi3)

,test_name,total_tests,abnormal_count,abnormal_rate_pct,avg_test_value,test_unit
0,Fasting Blood Glucose,5,4,80.0,131.00,mg/dL
1,Total Cholesterol,4,3,75.0,227.50,mg/dL
2,Serum Creatinine,5,2,40.0,1.66,mg/dL
3,Troponin-I,3,2,66.7,0.09,ng/mL
4,Hemoglobin,4,1,25.0,12.65,g/dL
5,White Blood Cell Count,3,1,33.3,9.17,10^3/uL


### KPI 4: Clinical Risk Stratification & High-Risk Patient Distribution

In [46]:
q4 = """
SELECT 
    clinical_risk_tier,
    COUNT(*) AS patient_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM patient_health_summary), 1) AS pct_of_cohort,
    ROUND(AVG(total_clinical_risk_score), 1) AS avg_risk_score,
    ROUND(AVG(avg_heart_rate), 1) AS cohort_avg_heart_rate,
    ROUND(AVG(avg_systolic_bp), 1) AS cohort_avg_systolic_bp
FROM patient_health_summary
GROUP BY clinical_risk_tier
ORDER BY avg_risk_score DESC;
"""
df_kpi4 = pd.read_sql_query(q4, conn)
display(df_kpi4)

,clinical_risk_tier,patient_count,pct_of_cohort,avg_risk_score,cohort_avg_heart_rate,cohort_avg_systolic_bp
0,High,6,40.0,8.2,103.2,156.0
1,Moderate,2,13.3,4.0,78.0,138.2
2,Low,7,46.7,0.6,72.1,119.7


### KPI 5: Patient Biometric Vitals Summary (Wearables)

In [47]:
q5 = """
SELECT 
    patient_id,
    patient_name,
    age,
    gender,
    avg_heart_rate,
    avg_systolic_bp,
    avg_diastolic_bp,
    min_spo2,
    clinical_risk_tier
FROM patient_health_summary
ORDER BY avg_heart_rate DESC;
"""
df_kpi5 = pd.read_sql_query(q5, conn)
display(df_kpi5.head(8))

,patient_id,patient_name,age,gender,avg_heart_rate,avg_systolic_bp,avg_diastolic_bp,min_spo2,clinical_risk_tier
0,P003,Robert Johnson,76,Male,118.0,171.7,105.0,89.0,High
1,P015,Charles Thompson,74,Male,114.0,140.0,89.0,95.0,High
2,P007,David Martinez,73,Male,107.7,159.0,98.0,90.0,High
3,P005,Michael Brown,61,Male,98.0,138.7,88.0,90.0,High
4,P011,William Jackson,56,Male,93.5,160.0,99.0,95.0,High
5,P009,James Anderson,83,Male,88.0,166.5,103.0,92.0,High
6,P008,Lisa Taylor,44,Female,86.0,125.0,81.0,94.0,Low
7,P001,John Doe,67,Male,80.0,142.3,92.3,96.0,Moderate


### KPI 6: Priority Action Roster - Patients Requiring Immediate Attention

In [48]:
q6 = """
SELECT 
    patient_id,
    patient_name,
    age,
    department,
    waiting_time_minutes,
    avg_heart_rate,
    avg_systolic_bp,
    min_spo2,
    abnormal_lab_count,
    total_clinical_risk_score,
    clinical_risk_tier,
    diagnosis,
    care_recommendation
FROM patient_health_summary
WHERE high_risk_flag = 1 
   OR waiting_time_minutes > 60
   OR abnormal_lab_count >= 2
ORDER BY total_clinical_risk_score DESC, waiting_time_minutes DESC;
"""
df_kpi6 = pd.read_sql_query(q6, conn)
display(df_kpi6)

# Close connection
conn.close()

,patient_id,patient_name,age,department,waiting_time_minutes,avg_heart_rate,avg_systolic_bp,min_spo2,abnormal_lab_count,total_clinical_risk_score,clinical_risk_tier,diagnosis,care_recommendation
0,P003,Robert Johnson,76,Cardiology,75.0,118.0,171.7,89.0,2,12,High,Unstable Angina,URGENT: Escalate to Senior Consultant / High-D...
1,P007,David Martinez,73,Cardiology,50.0,107.7,159.0,90.0,2,10,High,Congestive Heart Failure,URGENT: Escalate to Senior Consultant / High-D...
2,P009,James Anderson,83,Nephrology,65.0,88.0,166.5,92.0,2,9,High,Chronic Kidney Disease Stage 4,URGENT: Escalate to Senior Consultant / High-D...
3,P005,Michael Brown,61,Pulmonology,65.0,98.0,138.7,90.0,1,6,High,Acute COPD Exacerbation,URGENT: Escalate to Senior Consultant / High-D...
4,P015,Charles Thompson,74,Cardiology,45.0,114.0,140.0,95.0,0,6,High,Atrial Fibrillation,URGENT: Escalate to Senior Consultant / High-D...
5,P011,William Jackson,56,Cardiology,40.0,93.5,160.0,95.0,2,6,High,Hypertensive Urgency,URGENT: Escalate to Senior Consultant / High-D...
6,P001,John Doe,67,Cardiology,25.0,80.0,142.3,96.0,2,5,Moderate,Stage 2 Hypertension,PRIORITY: Continuous vitals monitoring & clini...


## 6. Conclusion & Interview Key Takeaways

1. **Full ETL Lifecycle**: Implemented modular phases (**Extract**, **Transform**, **Load**, and **Analyze**) with decoupled functions in `src/`.
2. **Data Cleansing Best Practices**: Handled missing data methodically with median and mode imputation, string normalization, and deduplication.
3. **Operational Metrics**: Successfully engineered waiting time metrics that highlight departmental bottlenecks (e.g. Cardiology & Pulmonology).
4. **Clinical Risk Modeling**: Formulated a standardized multi-parameter clinical early warning scoring engine that flags at-risk patients based on resting telemetry, lab values, and chronic comorbidities.
5. **Data Warehouse & SQL**: Modeled dimensional tables in SQLite with relational indexing and delivered actionable executive SQL queries.